# 04 · Panel Fixed Effects

Two-way FE (ship + month) removes time-invariant ship heterogeneity and common time trends. The coefficient should be much closer to the true β = 0.06.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

panel  = pd.read_parquet(DATA_DIR / "ship_month_panel.parquet")
events = pd.read_csv(DATA_DIR / "category_change_events.csv", parse_dates=["event_month"])

print(f"Panel: {panel.shape}  |  Events: {len(events)}")

In [ ]:
from linearmodels.panel import PanelOLS

## Prep: set panel index

In [ ]:
panel_idx = panel.set_index(["ship_id", "month"])
panel_idx["log_berth"] = np.log(panel_idx["berth_capacity"])

## Model 1 · Ship FE only (within estimator)

In [ ]:
fe1 = PanelOLS(
    panel_idx["log_revenue_per_berth"],
    panel_idx[["cabin_category_count", "log_berth"]],
    entity_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

coef = fe1.params["cabin_category_count"]
ci = fe1.conf_int().loc["cabin_category_count"]
print(f"β_ship_FE = {coef:.4f}  95% CI [{ci['lower']:.4f}, {ci['upper']:.4f}]")
print(f"True β    = 0.06")

## Model 2 · Two-way FE (ship + month) — preferred spec

In [ ]:
fe2 = PanelOLS(
    panel_idx["log_revenue_per_berth"],
    panel_idx[["cabin_category_count", "log_berth"]],
    entity_effects=True,
    time_effects=True,
).fit(cov_type="clustered", cluster_entity=True)

coef_twfe = fe2.params["cabin_category_count"]
ci_twfe = fe2.conf_int().loc["cabin_category_count"]
print(f"β_TWFE    = {coef_twfe:.4f}  95% CI [{ci_twfe['lower']:.4f}, {ci_twfe['upper']:.4f}]")
print(f"True β    = 0.06")
print()
print(fe2.summary.tables[1])

## Progression table: OLS → FE

In [ ]:
import statsmodels.formula.api as smf

m_ols = smf.ols("log_revenue_per_berth ~ cabin_category_count", data=panel).fit(cov_type="HC3")

estimates = [
    ("Naive OLS",         m_ols.params["cabin_category_count"]),
    ("Ship FE",           fe1.params["cabin_category_count"]),
    ("TWFE (ship+month)", fe2.params["cabin_category_count"]),
    ("True β",            0.06),
]

print(f"{'Estimator':<25} {'β':>8}  {'vs True β':>12}")
print("-" * 50)
for name, coef in estimates:
    ratio = f"{coef/0.06:.2f}×" if name != "True β" else "—"
    print(f"{name:<25} {coef:>8.4f}  {ratio:>12}")

## What fixed effects buy (and what they don't)

**Buy:** Removes all time-invariant ship-level confounding — latent scale, operator sophistication, baseline market position.

**Don't buy:** Doesn't handle *time-varying* confounders (e.g., if refits coincide with market-wide demand surges). The event study (notebook 08) addresses this by using the *timing* of refits as additional variation.

In [ ]:
print("""
Within-ship identification: FE uses the variation in cabin_category_count
WITHIN the same ship over time.

For treated ships: before refit → fewer categories; after refit → more.
For untreated ships: cabin_category_count is constant → contributes no variation.

This means the FE estimate is already close to an event study estimate, just without
the dynamic structure (leads/lags) that lets us test parallel trends.

TWFE coefficient = {:.4f}  (true β = 0.06)
Residual bias from FE: {:.1f}% above truth
""".format(coef_twfe, (coef_twfe - 0.06)/0.06 * 100))